# KADMON Optuna — QuickBundles + Partial OT

Les bundles sont compressés en centroïdes QuickBundles pondérés par la taille de leurs clusters, puis comparés avec Partial OT. Cette configuration sert de baseline tractographique pour évaluer les autres méthodes de compression. Le classement utilise `global_distance_mm`; pour chaque bundle, l'objectif minimise le rapport entre la distance intra-identité et la distance moyenne inter-identité.

## 1. Imports et configuration

In [1]:
from pathlib import Path
import os
import sys
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from optuna.trial import TrialState

WORKING_DIR = Path.cwd().resolve()
KADMON_ROOT = next(
    (path for path in (WORKING_DIR, *WORKING_DIR.parents) if (path / 'kadmon').is_dir() and (path / 'notebooks').is_dir()),
    None,
)
if KADMON_ROOT is None:
    raise RuntimeError(f'Racine KADMON introuvable depuis {WORKING_DIR}.')
NOTEBOOK_DIR = KADMON_ROOT / 'notebooks' / 'optuna'
BUNDLES_DIR = KADMON_ROOT / 'notebooks' / 'bundles'
STUDY_PATH = NOTEBOOK_DIR / 'studies' / 'optuna_reid.sqlite3'
if str(KADMON_ROOT) not in sys.path:
    sys.path.insert(0, str(KADMON_ROOT))

from kadmon.comparison import IneffectiveCompressionError, compare_bundles

optuna.logging.set_verbosity(optuna.logging.WARNING)
EXPERIMENT_NAME = 'quickbundles_partial'
COMPRESSION = 'quickbundles'
TRANSPORT = 'partial'
N_TRIALS = 80


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## 2. Données

Pour chaque bundle, l'acquisition `103818` est comparée aux dix acquisitions `_re`. `103818_re` constitue la comparaison intra-identité et les neuf autres acquisitions les comparaisons inter-identité.

In [2]:
SUBJECT_PAIRS = {
    '103818': '103818_re', '135528': '135528_re',
    '143325': '143325_re', '177746': '177746_re',
    '194140': '194140_re', '250427': '250427_re',
    '433839': '433839_re', '627549': '627549_re',
    '783462': '783462_re', '861456': '861456_re',
}
REFERENCE_SUBJECT = '103818'
INTRA_IDENTITY_SUBJECT = SUBJECT_PAIRS[REFERENCE_SUBJECT]
COMPARISON_SUBJECTS = tuple(SUBJECT_PAIRS.values())
SUBJECTS = (REFERENCE_SUBJECT, *COMPARISON_SUBJECTS)
N_POINTS = 12
SEED = 42
BUNDLES_TO_RUN = None  # tuple de noms exacts pour un essai court
N_JOBS_CPU = min(8, os.cpu_count() or 1)
MAX_COST_MATRIX_BYTES = 512 * 2**20  # 512 Mio par comparaison
MAX_REPRESENTATIVES = 5000  # Pruner avant MDF au-delà de cette limite

def index_subject(subject):
    directory = BUNDLES_DIR / subject / 'nn_8mm'
    suffix = f'_{N_POINTS}mpts_rasmm.npy'
    paths = sorted(directory.glob(f'*{suffix}'))
    return {path.name[:-len(suffix)]: path for path in paths}

SUBJECT_FILES = {subject: index_subject(subject) for subject in SUBJECTS}
bundle_names = sorted(set.intersection(*(set(index) for index in SUBJECT_FILES.values())))
if BUNDLES_TO_RUN is not None:
    missing = sorted(set(BUNDLES_TO_RUN) - set(bundle_names))
    if missing:
        raise ValueError(f'Bundles demandés absents : {missing}')
    bundle_names = [name for name in bundle_names if name in BUNDLES_TO_RUN]
if not bundle_names:
    raise RuntimeError('Aucun bundle commun au protocole de ré-identification.')

bundle_cache = {}
compression_cache = {}

def load_bundle(subject, bundle_name):
    key = (subject, bundle_name)
    if key not in bundle_cache:
        bundle = np.load(SUBJECT_FILES[subject][bundle_name], mmap_mode='r')
        if (bundle.ndim != 3 or bundle.shape[1:] != (N_POINTS, 3)
                or len(bundle) == 0 or not np.isfinite(bundle).all()):
            raise ValueError(f'Bundle invalide : {subject}/{bundle_name}, forme={bundle.shape}')
        bundle_cache[key] = bundle
    return bundle_cache[key]

print(f'Données : {BUNDLES_DIR}')
print('Protocole : 1 comparaison intra-identité et 9 inter-identité par bundle')
print(f'Bundles communs : {len(bundle_names)}; workers CPU : {N_JOBS_CPU}')
display(pd.DataFrame({'bundle': bundle_names}))


Données : /home/colin/Tractographie/KADMON/notebooks/bundles
Protocole : 1 comparaison intra-identité et 9 inter-identité par bundle
Bundles communs : 31; workers CPU : 8


,bundle
0,tractosearch_nn_8_0mm_all_AF_L_m
1,tractosearch_nn_8_0mm_all_AF_R_m
2,tractosearch_nn_8_0mm_all_CC_1_m
3,tractosearch_nn_8_0mm_all_CC_2a_m
4,tractosearch_nn_8_0mm_all_CC_2b_m
5,tractosearch_nn_8_0mm_all_CC_3_m
6,tractosearch_nn_8_0mm_all_CC_4_m
7,tractosearch_nn_8_0mm_all_CC_5_m
8,tractosearch_nn_8_0mm_all_CC_6_m
9,tractosearch_nn_8_0mm_all_CC_7_m


## 3. Espace de recherche

- `threshold`: 6 à 30 mm, par pas de 1 mm;
- `mass`: 0,50 à 1,00, par pas de 0,01.

Le seuil QuickBundles contrôle directement la compression : un seuil plus élevé regroupe davantage de streamlines et produit moins de centroïdes. `mass` fixe la fraction de masse empirique transportée par Partial OT.

In [3]:
def sample_parameters(trial):
    return (
        {'threshold': trial.suggest_float('threshold', 6.0, 30.0, step=1.0)},
        {'mass': trial.suggest_float('mass', 0.50, 1.00, step=0.01)},
    )


## 4. Objectif Optuna

In [4]:
def evaluate_bundle(bundle_name, compression_parameters, transport_parameters):
    source = load_bundle(REFERENCE_SUBJECT, bundle_name)
    pair_metrics = []
    for candidate_subject in COMPARISON_SUBJECTS:
        target = load_bundle(candidate_subject, bundle_name)
        result = compare_bundles(
            source, target, compression=COMPRESSION, transport=TRANSPORT,
            compression_parameters=compression_parameters,
            transport_parameters=transport_parameters,
            compression_cache=compression_cache,
            source_compression_key=(REFERENCE_SUBJECT, bundle_name),
            target_compression_key=(candidate_subject, bundle_name),
            max_cost_matrix_bytes=MAX_COST_MATRIX_BYTES,
            max_representatives=MAX_REPRESENTATIVES,
        )
        metrics = result['metrics']
        pair_metrics.append({
            'global_distance_mm': float(metrics['global_distance_mm']),
            'mean_displacement_mm': float(metrics['mean_mm']),
            'transported_mass': float(metrics['transported_mass']),
            'source_n_representatives': int(metrics['source_n_representatives']),
            'target_n_representatives': int(metrics['target_n_representatives']),
        })

    if len({row['source_n_representatives'] for row in pair_metrics}) != 1:
        raise RuntimeError(f'Compression source non reproductible pour {bundle_name}.')
    distances = np.asarray([row['global_distance_mm'] for row in pair_metrics])
    intra_index = COMPARISON_SUBJECTS.index(INTRA_IDENTITY_SUBJECT)
    intra_distance = float(distances[intra_index])
    inter_distances = np.delete(distances, intra_index)
    mean_inter_distance = float(inter_distances.mean())
    if not np.isfinite(mean_inter_distance) or mean_inter_distance <= 0:
        raise RuntimeError(f'Distance moyenne inter-identité invalide : {mean_inter_distance}')
    order = np.argsort(distances, kind='stable')
    return {
        'bundle': bundle_name,
        'intra_inter_ratio': intra_distance / mean_inter_distance,
        'intra_identity_top1_success': bool(int(np.argmin(distances)) == intra_index),
        'intra_identity_rank': int(np.flatnonzero(order == intra_index)[0] + 1),
        'intra_inter_separation_margin_mm': float(inter_distances.min() - intra_distance),
        'intra_identity_distance_mm': intra_distance,
        'mean_inter_identity_distance_mm': mean_inter_distance,
        'mean_displacement_mm': float(np.mean([r['mean_displacement_mm'] for r in pair_metrics])),
        'mean_transported_mass': float(np.mean([r['transported_mass'] for r in pair_metrics])),
        'mean_n_representatives': float(np.mean([v for r in pair_metrics for v in (r['source_n_representatives'], r['target_n_representatives'])])),
    }

def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    try:
        tasks = (delayed(evaluate_bundle)(name, compression_parameters, transport_parameters) for name in bundle_names)
        results = ([evaluate_bundle(name, compression_parameters, transport_parameters) for name in bundle_names]
                   if N_JOBS_CPU == 1 else
                   Parallel(n_jobs=min(N_JOBS_CPU, len(bundle_names)), backend='threading')(tasks))
    except IneffectiveCompressionError as exc:
        raise optuna.TrialPruned(str(exc)) from exc
    bundle_metrics = pd.DataFrame(results)
    valid = bundle_metrics['intra_identity_top1_success']
    aggregates = {
        'reid_valid_bundles': bundle_metrics.loc[valid, 'bundle'].tolist(),
        'reid_failed_bundles': bundle_metrics.loc[~valid, 'bundle'].tolist(),
        'mean_intra_inter_ratio': float(bundle_metrics['intra_inter_ratio'].mean()),
        'median_intra_inter_ratio': float(bundle_metrics['intra_inter_ratio'].median()),
        'intra_identity_top1_accuracy': float(valid.mean()),
        'mean_intra_identity_rank': float(bundle_metrics['intra_identity_rank'].mean()),
        'mean_intra_inter_separation_margin_mm': float(bundle_metrics['intra_inter_separation_margin_mm'].mean()),
        'mean_intra_identity_distance_mm': float(bundle_metrics['intra_identity_distance_mm'].mean()),
        'mean_inter_identity_distance_mm': float(bundle_metrics['mean_inter_identity_distance_mm'].mean()),
        'mean_displacement_mm': float(bundle_metrics['mean_displacement_mm'].mean()),
        'mean_transported_mass': float(bundle_metrics['mean_transported_mass'].mean()),
        'mean_n_representatives': float(bundle_metrics['mean_n_representatives'].mean()),
        'n_bundles': int(len(bundle_metrics)),
        'n_comparisons': int(len(bundle_metrics) * len(COMPARISON_SUBJECTS)),
        'elapsed_s': float(perf_counter() - started),
    }
    for name, value in aggregates.items():
        trial.set_user_attr(name, value)
    return aggregates['mean_intra_inter_ratio']


## 5. Optimisation

La base SQLite partagée est l'unique sortie automatique de l'étude.

In [5]:
STUDY_PATH.parent.mkdir(parents=True, exist_ok=True)
study = optuna.create_study(
    study_name=EXPERIMENT_NAME, storage=f'sqlite:///{STUDY_PATH}',
    load_if_exists=True, direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
completed_trials = sum(t.state == TrialState.COMPLETE for t in study.trials)
remaining_trials = max(0, N_TRIALS - completed_trials)
print(f'Étude : {completed_trials} essais terminés, {remaining_trials} à exécuter.')
if remaining_trials:
    study.optimize(
        objective, n_trials=remaining_trials, n_jobs=1,
        show_progress_bar=True, gc_after_trial=True,
        callbacks=[lambda study, trial: compression_cache.clear()],
    )
print(f'Base SQLite : {STUDY_PATH}')


Étude : 80 essais terminés, 0 à exécuter.
Base SQLite : /home/colin/Tractographie/KADMON/notebooks/optuna/studies/optuna_reid.sqlite3


## 6. Analyse des essais observés

In [6]:
trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
complete = trials_df[trials_df['state'] == 'COMPLETE'].dropna(subset=['value'])
display(complete.sort_values('value').head(10))

best = study.best_trial
display(pd.Series({'experiment': EXPERIMENT_NAME, 'trial': best.number,
                   'mean_intra_inter_ratio': best.value, **best.params,
                   **best.user_attrs}, name='meilleur essai').to_frame())
print('RE-ID validée :', best.user_attrs.get('reid_valid_bundles', []))
print('RE-ID échouée :', best.user_attrs.get('reid_failed_bundles', []))

optuna.visualization.plot_param_importances(study).show()
optuna.visualization.plot_slice(study).show()
optuna.visualization.plot_contour(study, params=['mass', 'threshold']).show()


,number,value,params_mass,params_threshold,user_attrs_elapsed_s,user_attrs_intra_identity_top1_accuracy,user_attrs_mean_displacement_mm,user_attrs_mean_inter_identity_distance_mm,user_attrs_mean_intra_identity_distance_mm,user_attrs_mean_intra_identity_rank,user_attrs_mean_intra_inter_ratio,user_attrs_mean_intra_inter_separation_margin_mm,user_attrs_mean_n_representatives,user_attrs_mean_transported_mass,user_attrs_median_intra_inter_ratio,user_attrs_n_bundles,user_attrs_n_comparisons,user_attrs_reid_failed_bundles,user_attrs_reid_valid_bundles,state
62,62,0.321797,0.90,29.0,13.968863,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
61,61,0.321797,0.90,29.0,13.981089,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
60,60,0.321797,0.90,29.0,14.029103,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
50,50,0.321797,0.90,29.0,14.502189,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
72,72,0.322464,0.88,29.0,13.962576,0.806452,4.617669,5.376563,1.843294,1.419355,0.322464,1.057421,2.285484,0.88,0.272980,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
85,85,0.322847,0.87,29.0,14.085155,0.806452,4.585263,5.323532,1.828686,1.419355,0.322847,1.027853,2.285484,0.87,0.275062,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
67,67,0.323176,0.91,29.0,14.073035,0.806452,4.732352,5.560567,1.903409,1.451613,0.323176,1.136918,2.285484,0.91,0.272599,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
68,68,0.323176,0.91,29.0,14.024792,0.806452,4.732352,5.560567,1.903409,1.451613,0.323176,1.136918,2.285484,0.91,0.272599,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
79,79,0.323176,0.91,29.0,13.963852,0.806452,4.732352,5.560567,1.903409,1.451613,0.323176,1.136918,2.285484,0.91,0.272599,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
77,77,0.325016,0.92,29.0,13.961480,0.806452,4.775845,5.629331,1.940630,1.451613,0.325016,1.149462,2.285484,0.92,0.268786,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE


,meilleur essai
experiment,quickbundles_partial
trial,50
mean_intra_inter_ratio,0.321797
threshold,29.0
mass,0.9
elapsed_s,14.502189
intra_identity_top1_accuracy,0.83871
mean_displacement_mm,4.691084
mean_inter_identity_distance_mm,5.495047
mean_intra_identity_distance_mm,1.875149


RE-ID validée : ['tractosearch_nn_8_0mm_all_AF_L_m', 'tractosearch_nn_8_0mm_all_AF_R_m', 'tractosearch_nn_8_0mm_all_CC_1_m', 'tractosearch_nn_8_0mm_all_CC_2a_m', 'tractosearch_nn_8_0mm_all_CC_2b_m', 'tractosearch_nn_8_0mm_all_CC_3_m', 'tractosearch_nn_8_0mm_all_CC_4_m', 'tractosearch_nn_8_0mm_all_CC_5_m', 'tractosearch_nn_8_0mm_all_CC_7_m', 'tractosearch_nn_8_0mm_all_CST_R_m', 'tractosearch_nn_8_0mm_all_ICP_L_m', 'tractosearch_nn_8_0mm_all_ICP_R_m', 'tractosearch_nn_8_0mm_all_IFOF_R_m', 'tractosearch_nn_8_0mm_all_ILF_L_m', 'tractosearch_nn_8_0mm_all_ILF_R_m', 'tractosearch_nn_8_0mm_all_MCP_m', 'tractosearch_nn_8_0mm_all_OR_L_m', 'tractosearch_nn_8_0mm_all_OR_R_m', 'tractosearch_nn_8_0mm_all_SLF_1_L_m', 'tractosearch_nn_8_0mm_all_SLF_1_R_m', 'tractosearch_nn_8_0mm_all_SLF_2_L_m', 'tractosearch_nn_8_0mm_all_SLF_2_R_m', 'tractosearch_nn_8_0mm_all_SLF_3_L_m', 'tractosearch_nn_8_0mm_all_SLF_3_R_m', 'tractosearch_nn_8_0mm_all_UF_L_m', 'tractosearch_nn_8_0mm_all_UF_R_m']
RE-ID échouée : ['tra

In [7]:
tradeoff_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
tradeoff_df = tradeoff_df[tradeoff_df["state"] == "COMPLETE"].dropna(subset=["value"]).rename(columns={
    "number": "trial", "value": "score",
    "params_threshold": "threshold", "params_mass": "mass",
    "user_attrs_mean_n_representatives": "n_representatives",
    "user_attrs_intra_identity_top1_accuracy": "reid_accuracy",
})
best_score = tradeoff_df["score"].min()
rows = []
for limit in (1, 2, 5):
    candidates = tradeoff_df[tradeoff_df["score"] <= best_score * (1 + limit / 100)]
    row = candidates.sort_values(["mass", "score"], ascending=[False, True]).iloc[0]
    rows.append([limit, int(row.trial), row.score, 100 * (row.score / best_score - 1), row.threshold, row.mass, row.n_representatives, row.reid_accuracy])
tradeoff = pd.DataFrame(rows, columns=["seuil (%)", "trial", "score", "écart relatif (%)", "threshold", "mass", "n_representatives", "reid_accuracy"])
display(tradeoff.style.format({"score": "{:.6f}", "écart relatif (%)": "{:.2f}", "threshold": "{:.0f}", "mass": "{:.2f}", "n_representatives": "{:.2f}", "reid_accuracy": "{:.0%}"}))


,seuil (%),trial,score,écart relatif (%),threshold,mass,n_representatives,reid_accuracy
0,1,67,0.323176,0.43,29,0.91,2.29,81%
1,2,65,0.327699,1.83,29,0.93,2.29,81%
2,5,47,0.336107,4.45,29,0.95,2.29,81%


## Interprétation des résultats

L'étude compte **80 essais `COMPLETE`**. Le **trial 50** est l'optimum strict observé (`score=0,321797`, `threshold=29 mm`, `mass=0,90`); les trials 60, 61 et 62 reproduisent exactement ce résultat. Son exactitude RE-ID Top-1 est de **83,87 %** (26 bundles sur 31), son rang intra-identité moyen est de **1,39** et sa marge moyenne de séparation intra–inter est de **1,11 mm**.

Le réglage transporte 90 % de la masse, mais QuickBundles ne produit en moyenne que **2,29 représentants par bundle**. Le seuil optimal est en outre très proche de la borne supérieure testée (`30 mm`). L'étude montre donc qu'une représentation globale extrêmement compacte favorise l'objectif RE-ID utilisé ici; elle ne démontre pas que deux centroïdes suffisent pour décrire des déviations anatomiques locales.

Parmi les essais à moins de 1 % de l'optimum continu, le trial 67 monte à `mass=0,91`, mais son Top-1 baisse à 80,65 %. Selon la règle RE-ID commune, le **trial 9** est le meilleur classement (`threshold=6 mm`, `mass=0,57`, Top-1 96,77 %, rang moyen 1,03 et environ 539 représentants). Pour l'usage anatomique de KADMON, le **trial 75** est toutefois retenu par défaut : `threshold=7 mm`, `mass=0,99`, Top-1 93,55 %, rang moyen 1,10, marge de séparation 2,00 mm et environ 296 représentants. Il transporte presque toute la masse tout en conservant une résolution substantielle.

- Pour la **meilleure RE-ID observée**, utiliser le trial 9 (`threshold=6 mm`, `mass=0,57`).
- Pour les **analyses anatomiques exécutées par défaut dans KADMON**, utiliser le trial 75 (`threshold=7 mm`, `mass=0,99`).
- Le trial 50 reste l'optimum du **ratio continu**, mais ses 2,29 représentants et son Top-1 de 83,87 % le rendent moins pertinent comme défaut.
- Le fait que l'optimum soit proche de la borne supérieure justifie une étude ciblée au-delà de 30 mm uniquement pour confirmer le plateau RE-ID; cela ne rendrait pas la représentation plus anatomiquement détaillée.
- Ces paramètres ne doivent pas être transférés à QuickBundles + Sinkhorn, qui possède son propre compromis entre compression et régularisation.